# EDA — Transaction Disputes (Unrecognized Charges)
### Factored AI & Data Hackathon 2026 — Team DataMastersGT

## What data was used and why

The full challenge dataset (`factored-datathon-2026-s3-157725502942-us-east-2-an`) has ~19M rows across 13 tables,
Hive-partitioned by date (`year=/month=/day=`). For this exploratory analysis **the full dataset was not downloaded**;
instead we built a sample targeted at the chosen use case (unrecognized-charge disputes), prioritizing temporal
coverage where it matters most:

| Table | Coverage downloaded | Rationale |
|---|---|---|
| `customers`, `products`, `branches`, `service_agents` | 100% (dimension tables, small) | Needed in full for joins |
| `complaints` | Full year 2026 (daily) | This is the project's target table — we need all the detail |
| `call_center_interactions` | Full year 2026 (daily) | Key table in the dispute flow (origin of the cases) |
| `call_transcripts` | May–June 2026 complete (daily) | To extract real intents/language; the full year is not needed for that |
| `satisfaction_surveys` | January–mid-Feb daily + weekly sample for the rest of the year | Supporting table (CSAT context), not the focus |
| `transactions` | Weekly sample April–June 2026 | Supporting/context table (very high volume, not the focus of the dispute flow) |

This sampling strategy (dense on the core tables of the use case, sparse on the context ones) was adopted
to keep the analysis manageable without losing the two tables that truly support the project: `complaints` and
`call_center_interactions`.

**Source:** official hackathon S3 bucket, read-only access provided in the challenge's Data Dictionary.
**Extraction date:** September 30, 2026.

## Setup

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

con = duckdb.connect()
base = "../hackathon-data"

con.execute(f"CREATE VIEW customers AS SELECT * FROM read_csv_auto('{base}/customers.csv', ignore_errors=true)")
con.execute(f"CREATE VIEW products AS SELECT * FROM read_csv_auto('{base}/products.csv', ignore_errors=true)")
con.execute(f"CREATE VIEW branches AS SELECT * FROM read_csv_auto('{base}/branches.csv', ignore_errors=true)")
con.execute(f"CREATE VIEW agents AS SELECT * FROM read_csv_auto('{base}/service_agents.csv', ignore_errors=true)")
con.execute(f"CREATE VIEW complaints AS SELECT * FROM read_csv_auto('{base}/complaints/**/*.csv', ignore_errors=true, union_by_name=true)")
con.execute(f"CREATE VIEW cci AS SELECT * FROM read_csv_auto('{base}/call_center_interactions/**/*.csv', ignore_errors=true, union_by_name=true)")
con.execute(f"CREATE VIEW surveys AS SELECT * FROM read_csv_auto('{base}/satisfaction_surveys/**/*.csv', ignore_errors=true, union_by_name=true)")
con.execute(f"CREATE VIEW txns AS SELECT * FROM read_csv_auto('{base}/transactions/**/*.csv', ignore_errors=true, union_by_name=true)")
con.execute(f"CREATE VIEW transcripts AS SELECT * FROM read_csv_auto('{base}/call_transcripts/**/*.csv', ignore_errors=true, union_by_name=true)")

print("Views created OK")

## 1. Sample size per table

In [ ]:
con.execute('''
SELECT 'customers' tbl, count(*) n FROM customers
UNION ALL SELECT 'products', count(*) FROM products
UNION ALL SELECT 'complaints', count(*) FROM complaints
UNION ALL SELECT 'call_center_interactions', count(*) FROM cci
UNION ALL SELECT 'satisfaction_surveys', count(*) FROM surveys
UNION ALL SELECT 'transactions (sample)', count(*) FROM txns
UNION ALL SELECT 'call_transcripts', count(*) FROM transcripts
''').df()

## 2. Key finding: "Transactions" disputes are almost 100% unrecognized charges

This directly validates the chosen use case: the dataset is already modeled almost exclusively
for the **unrecognized charge** scenario; it is not a generic category.

In [ ]:
con.execute('''
SELECT subcategory, count(*) n, round(avg(claimed_amount),0) avg_amt,
       round(100.0*sum(case when sla_breached then 1 else 0 end)/count(*),1) pct_breach,
       round(avg(resolution_days),1) avg_days,
       round(avg(resolution_satisfaction),2) avg_sat
FROM complaints WHERE category='Transactions' GROUP BY subcategory ORDER BY n DESC
''').df()

## 3. Complaints — overview by category

In [ ]:
con.execute('''
SELECT category, count(*) n, round(avg(claimed_amount),1) avg_amt,
       round(100.0*sum(case when sla_breached then 1 else 0 end)/count(*),1) pct_sla_breach,
       round(avg(resolution_days),1) avg_days
FROM complaints GROUP BY category ORDER BY n DESC
''').df()

## 4. Key finding: the current triage protects nothing

The SLA breach rate is practically the same regardless of the priority assigned to the case
(Critical, High, Medium, Low). This is the central evidence justifying a data-driven
re-prioritization agent instead of a manual/fixed assignment.

In [ ]:
con.execute('''
SELECT priority, count(*) n,
       round(100.0*sum(case when sla_breached then 1 else 0 end)/count(*),1) pct_breach,
       round(avg(resolution_days),1) avg_days
FROM complaints GROUP BY priority ORDER BY priority
''').df()

## 5. Status and reception channel of the disputes

In [ ]:
con.execute('''
SELECT status, count(*) n FROM complaints GROUP BY status ORDER BY n DESC
''').df()

In [ ]:
con.execute('''
SELECT reception_channel, count(*) n FROM complaints GROUP BY reception_channel ORDER BY n DESC
''').df()

## 6. Broken traceability: 0% of Transactions disputes are linked to the call that originated them

In [ ]:
con.execute('''
SELECT origin_interaction_id IS NOT NULL as has_origin_interaction, count(*) n
FROM complaints WHERE category='Transactions' GROUP BY 1
''').df()

## 7. Call Center Interactions — resolution and escalation rate by reason type

"Complaint"-type interactions resolve on first contact at less than half the rate of the rest
(43.3% vs. 90%+), consistent with them being the ones that most often end up escalating to `complaints`.

In [ ]:
con.execute('''
SELECT reason_category, count(*) n,
       round(100.0*sum(case when was_resolved then 1 else 0 end)/count(*),1) pct_first_contact_resolved,
       round(100.0*sum(case when was_escalated then 1 else 0 end)/count(*),1) pct_escalated
FROM cci GROUP BY reason_category ORDER BY n DESC
''').df()

## 8. Transcripts — confirms the dataset's language limitation

100% of the transcripts detected in the sample are in Spanish. There is no Portuguese data
in the dataset — this is documented as a known limitation of the *data*, not of the system: the Portuguese
test cases for the agent are generated manually by the team.

In [ ]:
con.execute('''
SELECT detected_language, count(*) n FROM transcripts GROUP BY detected_language ORDER BY n DESC
''').df()

### Intent detected in the transcripts

The `detected_intents` field that already ships with the dataset is almost entirely `"consulta_general"` (general inquiry) — too generic
to route a case. This justifies building our own intent classifier (vs. that raw field).

In [ ]:
con.execute('''
SELECT detected_intents, count(*) n FROM transcripts GROUP BY detected_intents ORDER BY n DESC LIMIT 10
''').df()

## 9. Customer satisfaction (surveys)

In [ ]:
con.execute('''
SELECT survey_type, round(avg(main_score),2) avg_score, min(main_score) min_score, max(main_score) max_score, count(*) n
FROM surveys GROUP BY survey_type ORDER BY n DESC
''').df()

## 10. Data quality — limitations to keep in mind in the pipeline

They are not hidden, they are documented: the challenge rewards acknowledging them and handling them explicitly.

In [ ]:
con.execute('''
SELECT
  count(*) total_complaints,
  sum(case when claimed_amount is null then 1 else 0 end) nulls_claimed_amount,
  round(100.0*sum(case when claimed_amount is null then 1 else 0 end)/count(*),1) pct_nulls_amount,
  sum(case when resolution_date is null then 1 else 0 end) nulls_resolution_date,
  round(100.0*sum(case when resolution_date is null then 1 else 0 end)/count(*),1) pct_nulls_resdate
FROM complaints
''').df()

The nulls in `claimed_amount` and `resolution_date` are consistent with ~68% of the cases
still being `In Process`/`Open` (not yet closed) — realistic for a bank in operation, but the
agent's pipeline must handle those nulls without assuming there is always an amount or closing date.

## Conclusions — what these findings justify for the project

1. **Real, measurable problem**: 2,093 unrecognized-charge disputes/year, $2,458 average claimed, 22% breach the SLA, 15 days average resolution.
2. **Manual triage does not work** (SLA breach ~20-22% even across all priorities) → justifies a risk-based re-prioritization model, compared against that fixed-rules baseline.
3. **The dataset's intent field is too generic** (95% "consulta_general") → justifies our own intent classifier.
4. **Interaction→dispute traceability is broken** (0% linked) → the system solves it by design.
5. **The dataset is 100% Spanish** → the Portuguese cases for the agent are generated manually and documented as a data limitation, not a system limitation.